# FairGate Studio

**Before the first run:** Runtime → Change runtime type → **T4 GPU**, and add your GitHub token as a
Colab secret named `GITHUB_TOKEN` (key icon on the left, with Notebook access on).

**Then: Runtime → Run all.** That's it. The first launch takes about 8 minutes; later launches load the
saved 3-bit model from Google Drive and are faster. Each start-up step prints how long it took, and
the app's header says whether the live models match the saved results.

You don't need to restart the session after the install: the app runs as its own process.

In [ ]:
#@title ▶ Launch FairGate Studio (live, on the GPU)
USE_DRIVE = True   #@param {type:"boolean"}
BRANCH = "main"    #@param {type:"string"}
# USE_DRIVE keeps a copy of the 3-bit model on your Google Drive, so every launch after the first
# skips quantizing (about 2 minutes saved). Colab asks you once to allow Drive access.

import os, subprocess
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
repo = f"https://{token}@github.com/GraverSurvivor/ELSEM3.git"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***").strip()
    if r.returncode and out:
        print(out)
    return r.returncode

print("1/3  Getting the code...")
if not os.path.exists("/content/ELSEM3"):
    sh(f"git clone -q --branch {BRANCH} {repo} /content/ELSEM3")
else:  # a re-run in the same session picks up the latest code
    sh(f"cd /content/ELSEM3 && git fetch -q {repo} {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD")
os.chdir("/content/ELSEM3")

print("2/3  Installing (about 1 minute)...")
sh('pip install -q -r requirements.txt "gradio>=5" "transformers==5.0.0" "hqq==0.2.8.post1" '
   '"accelerate==1.13.0" "tokenizers==0.22.2" "safetensors==0.7.0"')

cache = ""
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = "--cache-dir /content/drive/MyDrive/ELSEM3_studio_cache"

print("3/3  Starting the app. The public link (https://....gradio.live) appears below when it is ready.")
!python -m src.product.studio --share {cache}

If the GPU session dies or there's no GPU available, stop the cell above and run this one instead.

In [ ]:
#@title Backup: the app on saved results, no GPU needed
import os
os.chdir("/content/ELSEM3")
!python -m src.product.studio --offline --share